# 🔍 **Why a Schema-Driven, CI/CD-Powered Evaluation Pipeline Matters**

Welcome to **Lab 3**. Before we crack open any notebooks, let’s set the stage:

Modern AI initiatives rarely belong to a single team. Product managers sketch user journeys, search engineers tune retrieval, clinicians vet medical facts, privacy officers hunt PHI, and DevOps keep everything humming in prod. A **schema-centric evaluation pipeline** lets *all* of those disciplines propose, review, and extend tests without touching Python files or service code.  

Because every rule lives in YAML—metrics to run, datasets to use, parameters to tweak—any stakeholder can iterate safely in a pull request, and the exact state of the system at *any* commit is reproducible on demand.

## 🧩 A Schema Is the Enabler for Multidisciplinary Teams  
AI systems are constellations of subsystems—vector search, OCR, planners, LLM reasoning, safety filters. Hand-offs break when teams use different dialects:

* **Search** exports `results.json`; **planner** expects `rankings.jsonl`.  
* Yesterday’s `ndcg_10` becomes `NDCG@10` and dashboards explode.

In **MedEvals**, every evaluation lives in YAML. The file freezes contracts—class names, evaluator lists, required artifacts—long before code runs. Break the contract and CI barks immediately. The code is focused on enabling and rationalizing the interfaces.

## 🔄 Deterministic Pipelines Keep “Yesterday’s Good” From Becoming “Tomorrow’s Mystery”  
Re-running an evaluation should feel like re-running a unit test: same inputs → same outputs. MedEvals’ `PipelineEvaluator` enforces that discipline:

1. **Preprocess** — load YAML, spin up evaluators, stage data.  
2. **Evaluate** — kick off each evaluator (pure functions, no hidden RNG).  
3. **Post-process** — write metrics to content-addressable paths such as `artifacts/<git_sha>/metrics.json`.

With commit-plus-dataset hashes embedded in every artifact, any metric can be traced to the exact code + data snapshot that produced it.

## 🌐 Holistic Evaluation ≠ One Metric to Rule Them All  
An end-user experience can fail in countless ways. MedEvals lets each discipline bolt in its own evaluator block—no tangled imports, no edits to pipeline code. That means for each and every application there are a host of concerns across the evaluation and across disciplines:

| Discipline / Role           | Question They Must Answer                              | Example Metric                       |
|-----------------------------|--------------------------------------------------------|--------------------------------------|
| **Search Engineers**        | “Did we surface the *right* docs?”                    | NDCG@k, Recall@k                     |
| **Vision/OCR Specialists**  | “Are we leaking PHI?”                                 | Word-level accuracy, leakage rate    |
| **Task-Planner Authors**    | “Is the plan transparent?”                            | Goal-coverage %, step validity       |
| **RAG Chat Curators**       | “Is the answer grounded & polite?”                    | Factuality %, toxicity score         |
| **Clinical SMEs**           | “Is medical advice safe and guideline-compliant?”     | Guideline adherence %, contraindication hits |
| **Privacy & Compliance**    | “Are we exposing protected data?”                     | PII/PHI recall & precision           |
| **Product & QA**            | “Does the user journey stay inside SLA?”              | Latency p95, error budgets           |

Add a new evaluator in YAML, adjust args, and the framework should do the rest.

## ⚙️ CI/CD Turns Insight Into Guardrails  
Manual spreadsheet checks don’t scale. Once evaluators are codified:

1. **Every PR** fires a fast smoke-test subset.  
2. **Nightly** jobs run the full benchmark; metric drift beyond tolerance can trigger alerts such as into Teams, Email, and your favorite systems.
3. Dashboards and table views can be sliced and diced by tags like `pipeline=clinicalExtractor` or `commit=<git_sha>` so a team can spot regressions before users do.

MedEvals wires this up out of the box—tags live next to metrics, eval IDs track scenarios, YAML drives everything.

## ✨ Net Result 

* **Developers & MLEs** catch relevance or latency regressions immediately.  
* **Data Scientists & Generative AI Developers** iterate safely—new rerankers or new prompts and don't push them until the metrics match or meet the benchmarks.  
* **Clinicians & SMEs** add domain tests without writing code.  
* **Platform & DevOps** reproduce any run from any commit in minutes.

That same schema-centric, deterministic, CI-enforced philosophy at the heart of **MedEvals** is what this lab notebook will teach you to extend. Let’s dive in and make evaluations first-class citizens of our CI/CD universe.


Now consider what our schema will need to look like. Your organization may define things differently, but the thought process remains the same: we should define what our evaluations should look like holistically and they should be defined. But what goes into an evaluation?

As we saw in AI Foundry, we need each and every evaluation to have a row populated with additional data to evaluate: query, response, ground_truth, or maybe additional context. Then these are passed to the evaluators that are configured, and output metrics with the data.

But to generate this dataset may require parameters of it's own, and it's processing will certainly need to be catalogued as state. It **matters** on how the dataset is generated just as much as the evaluations themselves; we want holistic evaluations that pairs with reproducible research.

For this lab consider this schema that we put together, consider what each component includes:

```yaml
# ==============================================================================
# Test Evaluation Configuration
# ==============================================================================
# This configuration defines a policy‐retrieval evaluation test.
# It specifies:
#   - A descriptive overview of the evaluation.
#   - A disclaimer regarding any special conditions or limitations.
#   - The retrieval pipeline to run (class plus parameter files).
#   - A list of evaluator definitions (each with name and class).
#   - The test case identifier(s) to execute.
# ==============================================================================

policy-retrieval-hybrid-semantic-001:
  # Description: A human-readable explanation of the evaluation purpose.
  description: >
    Evaluation for policy retrieval against AI Search,
    focusing on the queries against the payor policy of Cigna.
  # Disclaimer: Notes any limitations or special conditions.
  disclaimer: >
    Evaluations are performed zero-shot without additional fine-tuning.
  # Pipeline: Defines the retrieval evaluation pipeline details.
  pipeline:
    # class: Fully qualified class name of the retrieval evaluator.
    class: src.pipeline.policyIndexer.evaluator.PolicyIndexerEvaluator
    # params: Parameters for the pipeline execution.
    params:
      # qrels: Path to the ground-truth relevance judgments.
      qrels: "evals/benchmark/medindexer/qrels.jsonl"
      # rankings: Path to the search results to be evaluated.
      rankings: "evals/benchmark/medindexer/rankings-hybrid-semantic.jsonl"
  # Evaluators: A list of metrics to apply to the retrieval results.
  evaluators:
    - name: "NDCGEvaluator"           # Identifier for the NDCG metric evaluator.
      class: src.evals.custom.ndcg_evaluator:NDCGEvaluator

    - name: "SearchRecallEvaluator"   # Identifier for the Recall@K metric evaluator.
      class: src.evals.custom.search_recall_evaluator:SearchRecallEvaluator
  # Cases: References to the case‐specific configuration(s) to run.
  cases:
    - policy-retrieval-hybrid-semantic-001.v0

# ==============================================================================
# Case‐Specific Configuration (Example: policy-retrieval-hybrid-semantic-001.v0)
# ==============================================================================
# This section provides details for one test case:
#   - metrics: Which evaluators (by name) should produce outputs.
#   - evaluations: A list of query strings mapped to their qid.
# ==============================================================================
policy-retrieval-hybrid-semantic-001.v0:
  # Metrics: List of evaluator names (must match definitions above).
  metrics: [NDCGEvaluator, SearchRecallEvaluator]
  # Evaluations: Each item includes:
  #   - query: The search query to evaluate.
  #   - qid:   The identifier that ties back to the qrels file.
  evaluations:
    - query: "Crohn's disease Adalimumab prior authorization criteria"
      qid: q1
    - query: "Adalimumab treatment guidelines for Crohn's disease"
      qid: q2
    - query: "Prior authorization requirements for Adalimumab in inflammatory bowel disease"
      qid: q3
    - query: "Adalimumab therapy approval for Crohn's disease"
      qid: q4
    - query: "Inflammatory bowel disease Adalimumab authorization policy"
      qid: q5
    - query: "Adalimumab usage criteria for Crohn's disease"
      qid: q6
    - query: "Crohn's Adalimumab medication authorization guidelines"
      qid: q7
```

Go through the components and understand how the logic flows. We have saved this example to `evals/cases/policy-retrieval-hybrid-semantic-001.yaml` to be used. Notice how not all of our evaluation rows have the data required - this information is expected to be included by our pipeline `PolicyRetrievalEvaluator`. Search stores can change as indexing happens, so it is reasonable to expect those to not be defined and to be dynamically created; in support of reproducible research we would suggest a versioning strategy of the index and for that to be included on any tags on the evaluation.

We're going to start by defining our pipeline, breaking down into 3 components:

1. **preprocessing** - this is where we intend on creating our evaluation dataset to comply with AI Foundry's schema evaluation, in addition to our evaluators. Automation can be included here to generate the dataset on the fly, including search retrievals and rankings and chat responses.
2. **evaluations** - this is where the actual evaluations take place powered by AI Foundry's evaluate method, and we can choose for local or cloud approach like we have in the past.
3. **postprocessing** - this is where we can take the generated metrics from the evaluations and use that to generate additional artifacts or handle additional custom processing.

We put together the first pass of what a pipeline could look on behalf of the search retrieval below. First start with setting the `PYTHONPATH` as needed:

In [1]:
import os, sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

Now create the class for the pipeline:

In [35]:
import os
import yaml
import json
import subprocess
import importlib
import inspect
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.evaluation import evaluate
from azure.ai.evaluation._evaluate._eval_run import EvalRun
import src.evals.sdk.custom_azure_ai_evaluations as custom_eval
from src.aifoundry.aifoundry_helper import AIFoundryManager
from src.evals.case import Case

# hook custom start for AI Foundry
EvalRun._start_run = custom_eval.custom_start_run

class PolicyRetrievalEvaluator:
    """
    Self-contained evaluator that:
      • Uses ROOT_DIR = repo root (one level up from cwd)
      • Scans ROOT_DIR/evals/cases/*.yaml
      • Filters only those whose pipeline.class == EXPECTED_PIPELINE
      • Generates automation_eval_data_<rank-type>.jsonl under ROOT_DIR
      • Instantiates evaluators per YAML
      • Exposes preprocess(), run_evaluations(), post_processing()
    """
    EXPECTED_PIPELINE = "src.pipeline.policyIndexer.evaluator.PolicyIndexerEvaluator"
    DEFAULT_ARGS_BY_EVALUATOR = {
        "NDCGEvaluator":         {"k": 10},
        "SearchRecallEvaluator": {"k": 10},
    }

    def __init__(self):
        load_dotenv()
        self.ROOT_DIR           = Path.cwd().parent
        self.cases              = {}
        self.ai_foundry_manager = AIFoundryManager()

    def _load_yaml(self, path: str) -> dict:
        try:
            with open(path, "r") as f:
                return yaml.safe_load(f) or {}
        except Exception:
            return {}

    def _find_pipeline_root(self, cfg: dict):
        for key, val in cfg.items():
            if isinstance(val, dict) and "pipeline" in val:
                return key, val
        raise RuntimeError("No 'pipeline' entry found")

    def _resolve_object(self, value: str):
        try:
            mod_path, obj_path = value.split(":", 1)
            module = importlib.import_module(mod_path)
            obj = module
            for part in obj_path.split("."):
                obj = getattr(obj, part)
            return obj
        except Exception:
            return value

    def _get_git_hash(self) -> str:
        if os.environ.get("GIT_HASH"):
            return os.environ["GIT_HASH"]
        try:
            return subprocess.check_output(
                ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.STDOUT
            ).decode().strip()
        except Exception:
            return "unknown"

    def _generate_custom_tags(self, case_id: str, git_commit: str):
        tags = [("case", case_id), ("commit", git_commit)]
        custom_eval.CUSTOM_TAGS = tags
        return tags

    def _instantiate_evaluators(self, root: dict) -> dict:
        evaluators = {}
        for ev in root.get("evaluators", []):
            name     = ev["name"]
            cls_path = ev["class"]
            module_path, cls_name = cls_path.split(":", 1)
            module = importlib.import_module(module_path)
            cls    = getattr(module, cls_name)
            args   = dict(ev.get("args", {}))

            sig     = inspect.signature(cls.__init__)
            missing = [
                p.name for p in sig.parameters.values()
                if p.name not in ("self","model_config")
                   and p.default is inspect._empty
                   and p.name not in args
            ]
            if missing:
                if name in self.DEFAULT_ARGS_BY_EVALUATOR:
                    defaults = self.DEFAULT_ARGS_BY_EVALUATOR[name]
                    args.update(defaults)
                    print(f"[auto-fill] {name}: using defaults {defaults} for missing {missing}")
                else:
                    raise ValueError(f"Evaluator '{name}' missing args for: {missing}")

            if "model_config" in sig.parameters and "model_config" not in args:
                mc = {
                    "azure_endpoint":   os.getenv("AZURE_OPENAI_ENDPOINT"),
                    "api_key":          os.getenv("AZURE_OPENAI_KEY"),
                    "azure_deployment": os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_ID"),
                    "api_version":      os.getenv("AZURE_OPENAI_API_VERSION"),
                }
                if any(v is None for v in mc.values()):
                    raise ValueError("Missing AZURE_OPENAI_* env vars")
                args["model_config"] = mc

            final_args = {
                k: (self._resolve_object(v) if isinstance(v, str) and ":" in v else v)
                for k, v in args.items()
            }
            evaluators[name] = cls(**final_args)
        return evaluators

    def _generate_automation_eval_data(self, yaml_path: Path) -> Path:
        cfg      = self._load_yaml(str(yaml_path))
        root_key, root = self._find_pipeline_root(cfg)
        params  = root["pipeline"]["params"]

        qrels_f = self.ROOT_DIR / params["qrels"]
        rank_f  = self.ROOT_DIR / params["rankings"]
        if not qrels_f.exists() or not rank_f.exists():
            raise RuntimeError(f"Missing files: {qrels_f}, {rank_f}")

        qrels = {}
        for ln in open(qrels_f, "r"):
            e   = json.loads(ln)
            qid = e.get("qid") or e.get("query")
            doc = e.get("document") or e.get("doc_id")
            rel = int(e.get("relevant", e.get("relevance_score", 0)))
            qrels.setdefault(qid, {})[doc] = rel

        rank_type = rank_f.stem.replace("rankings-", "")
        out_path  = rank_f.parent / f"automation_eval_data_{rank_type}.jsonl"

        with open(rank_f, "r") as fin, open(out_path, "w") as fout:
            for ln in fin:
                rec = json.loads(ln)
                qid = rec.get("query_id") or rec.get("query")
                fout.write(json.dumps({
                    "query_id":     qid,
                    "response":     {"ranking": rec["ranking"]},
                    "ground_truth": qrels.get(qid, {})
                }) + "\n")

        print(f"Generated eval data file: {out_path.relative_to(self.ROOT_DIR)}")
        return out_path

 async def preprocess(self):
    cases_dir = self.ROOT_DIR / "evals" / "cases"
    print(f"Scanning for YAML configs in: {cases_dir}")
    for yaml_path in cases_dir.glob("*.yaml"):
        print(f"Found YAML: {yaml_path.relative_to(self.ROOT_DIR)}")
        cfg = self._load_yaml(str(yaml_path))
        try:
            root_key, root = self._find_pipeline_root(cfg)
        except RuntimeError:
            print(f"  → Skipping {yaml_path.name}: no pipeline section")
            continue

        pipeline_cls = root["pipeline"].get("class")
        if pipeline_cls != self.EXPECTED_PIPELINE:
            print(f"  → Skipping {yaml_path.name}: pipeline.class={pipeline_cls}")
            continue

        # iterate _all_ declared case names (or default to the root_key)
        for case_name in root.get("cases", [root_key]):
            print(f"  → Loading case '{case_name}'")
            eval_data  = self._generate_automation_eval_data(yaml_path)
            evals_dict = self._instantiate_evaluators(root)

            case = Case(case_name=case_name)
            case.evaluators = evals_dict

            def cm():
                class Ctx:
                    def __enter__(self_inner):
                        return str(eval_data)
                    def __exit__(self_inner, *args):
                        pass
                return Ctx()

            case.create_evaluation_dataset = cm
            self.cases[case_name] = case

    print(f"Total cases loaded: {len(self.cases)}")

    async def run_evaluations(self):
        git_hash = self._get_git_hash()
        for cid, case in self.cases.items():
            print(f"Running evaluation for case '{cid}'")
            ev_cfg = {
                name: {"column_mapping": {
                    "response":     "${data.response}",
                    "ground_truth": "${data.ground_truth}",
                    "query_id":     "${data.query_id}"
                }}
                for name in case.evaluators
            }
            with case.create_evaluation_dataset() as ds:
                custom_eval.CUSTOM_TAGS = self._generate_custom_tags(cid, git_hash)
                result = evaluate(
                    evaluation_name   = f"{cid}-{git_hash}",
                    data              = ds,
                    evaluators        = case.evaluators,
                    evaluator_config  = ev_cfg,
                    azure_ai_project  = self.ai_foundry_manager.project_config,
                )
                case.azure_eval_result = result
                print(f"  → Completed '{cid}': metrics {result.get('metrics', {})}")

    def post_processing(self) -> dict:
        print("Post-processing results:")
        return {
            "cases": [
                {"case": cid, "results": case.azure_eval_result}
                for cid, case in self.cases.items()
            ]
        }

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 156)

The above pipeline code sets us up for processing and we can now invoke it by starting the pipeline. Do so as follows:

In [36]:
evaluator = PolicyRetrievalEvaluator()

2025-04-27 17:29:40,138 - AIFoundryManager - MainProcess - INFO     Configuration validation successful. (aifoundry_helper.py:_validate_configurations:58)
INFO:AIFoundryManager:Configuration validation successful.
2025-04-27 17:29:40,140 - AIFoundryManager - MainProcess - INFO     AIProjectClient initialized successfully. (aifoundry_helper.py:_initialize_project:103)
INFO:AIFoundryManager:AIProjectClient initialized successfully.


In a perfect world we want to run the automation completely end to end, but for this lab we are breaking down the evaluation steps into our components. Let's start with preprocessing, where the evaluators are initialized on our local machine in addition to the dataset for evaluation created.

In [37]:
await evaluator.preprocess()

Scanning for YAML configs in: /Users/marcjimz/Documents/Development/aihlsignited-medevals/evals/cases
Found YAML: evals/cases/policy-retrieval-hybrid-semantic-001.yaml
  → Loading case 'policy-retrieval-hybrid-semantic-001'
Generated eval data file: evals/benchmark/medindexer/automation_eval_data_hybrid-semantic.jsonl
[auto-fill] NDCGEvaluator: using defaults {'k': 10} for missing ['k']
[auto-fill] SearchRecallEvaluator: using defaults {'k': 10} for missing ['k']
Total cases loaded: 1


With the evaluation data generated, we can now run the evaluations. We should expect at completion to see both the output from the execution in the notebook but also the results uploaded to AI Foundry.

Let's run it now:

In [38]:
await evaluator.run_evaluations()

[2025-04-27 17:29:44 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-27 17:29:44 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_ndcg_evaluator_ndcgevaluator_u4qmwu8o_20250427_172944_850739, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_u4qmwu8o_20250427_172944_850739/logs.txt
[2025-04-27 17:29:44 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-27 17:29:44 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_search_recall_evaluator_searchrecallevaluator_nx_7bc99_20250427_172944_850814, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_search_recall_evaluator_searchrecallevaluator

Running evaluation for case 'policy-retrieval-hybrid-semantic-001'
2025-04-27 17:29:44 -0700   45223 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-27 17:29:44 -0700   45223 execution.bulk     INFO     The timeout for the batch run is 3600 seconds.
2025-04-27 17:29:44 -0700   45223 execution.bulk     INFO     Current system's available memory is 9154.28125MB, memory consumption of current process is 317.203125MB, estimated available worker count is 9154.28125/317.203125 = 28
2025-04-27 17:29:44 -0700   45223 execution.bulk     INFO     Set process count to 4 by taking the minimum value among the factors of {'default_worker_count': 4, 'row_count': 35, 'estimated_worker_count_based_on_memory_usage': 28}.
2025-04-27 17:29:46 -0700   45223 execution.bulk     INFO     Process name(SpawnProcess-34)-Process id(54320)-Line number(0) start execution.
2025-04-27 17:29:46 -0700   45223 execution.bulk     INFO     Process nam

Now that it's finally done, we can run the final post processing step to conclude with our summary:

In [39]:
summary = evaluator.post_processing()
print(summary)

Post-processing results:
{'cases': [{'case': 'policy-retrieval-hybrid-semantic-001', 'results': {'rows': [{'inputs.query_id': 'q1', 'inputs.response': {'ranking': {'d40': 2.873411893844604, 'd52': 2.826622009277343, 'd32': 2.8110253810882573, 'd66': 2.753601312637329, 'd17': 2.720281362533569, 'd14': 2.711065053939819, 'd51': 2.703975677490234, 'd18': 2.667110919952392, 'd2': 2.660730600357055, 'd71': 2.613231658935547, 'd50': 2.594090223312378, 'd34': 2.579911470413208, 'd1': 2.557225465774536, 'd41': 2.419337034225464, 'd25': 2.412779331207275, 'd3': 2.397005558013916, 'd7': 2.375914573669433, 'd15': 2.363330841064453, 'd31': 2.335416316986084, 'd44': 2.331694602966308, 'd21': 2.329656362533569, 'd67': 2.28756308555603, 'd8': 2.170100927352905, 'd60': 2.093313932418823, 'd23': 2.078780651092529, 'd33': 1.9138640165328982, 'd56': 1.89064621925354, 'd48': 1.83977997303009, 'd61': 1.812840342521667, 'd57': 1.6710525751113892, 'd27': 1.652620196342468, 'd28': 1.4980716705322261, 'd9': 1.

This example ran on the above of one test case. What if you wanted to run your own? Feel free to add your own case to the `evals/cases` folder, and watch it be evaluated and executed against with AI Foundry. All you need to do is define the schema using the YAML structure, reference the queries we previously defined. 

Create a new file `policy-retrieval-vector-001.yaml` with this as the contents:

```yaml
policy-retrieval-vector-001:
  description: >
    Evaluation for policy retrieval against AI Search, focusing on the queries against the payor policy of Cigna.
  disclaimer: >
    Evaluations are performed zero-shot without additional fine-tuning.
  pipeline:
    class: src.pipeline.policyIndexer.evaluator.PolicyIndexerEvaluator
    params:
      qrels: "evals/benchmark/medindexer/qrels.jsonl"
      rankings: "evals/benchmark/medindexer/rankings-vector.jsonl"
  evaluators:
    - name: "NDCGEvaluator"
      class: src.evals.custom.ndcg_evaluator:NDCGEvaluator
    - name: "SearchRecallEvaluator"
      class: src.evals.custom.search_recall_evaluator:SearchRecallEvaluator
  cases:
    - policy-retrieval-vector-001.v0

policy-retrieval-vector-001.v0:
  metrics: [NDCGEvaluator, SearchRecallEvaluator]
  evaluations:
    - query: "Crohn's disease Adalimumab prior authorization criteria"
      qid: q1
    - query: "Adalimumab treatment guidelines for Crohn's disease"
      qid: q2
    - query: "Prior authorization requirements for Adalimumab in inflammatory bowel disease"
      qid: q3
    - query: "Adalimumab therapy approval for Crohn's disease"
      qid: q4
    - query: "Inflammatory bowel disease Adalimumab authorization policy"
      qid: q5
    - query: "Adalimumab usage criteria for Crohn's disease"
      qid: q6
    - query: "Crohn's Adalimumab medication authorization guidelines"
      qid: q7
```

We could further add additional use cases for other queries for other payor policies. Feel free to experiment and add your own!

Then run the following lines again, uncommenting as needed.

In [9]:
# await evaluator.preprocess()
# await evaluator.run_evaluations()
# summary = evaluator.post_processing()

Before we can move onto CI/CD automations, we now need to incorporate strict benchmarks into our evaluations. While our evaluations generate metrics across the board, there is still benchmarking requireed in collaboration with our holistic review of an application to go to production.

For example, we may have several use cases across the board but with different needs. Metrics that measure deterministic accuracy may be much stricter, such as the approval or denial of a reviewed claim, but others may have minimum thresholds based on custom metrics such as one that measures the number of factual claims made as it relates to a ground truth.

When choosing a benchmark, it's important that these are made in collaboration with your business sponsors, to protect and enable your business. Failing a benchmark should be explainable, and a benchmark should only be updated should it align with your application expectations.

We can use the `PyTest` framework to asset some of the benchmarks we created here today. You can run this now:


In [18]:
import ipytest

We run the PyTest modules inline, but traditionally these will be saved to `tests/test*.py` files and executed when running the `pytest` command.

In [41]:
%%ipytest
import pytest
import asyncio
import json
import nest_asyncio

# Patch the loop so nested run_until_complete calls are allowed
nest_asyncio.apply()

@pytest.fixture(scope="session")
def med_indexer_summary():
    """
    Runs the PolicyRetrievalEvaluator pipeline once (sync) and yields its parsed summary.
    """
    evaluator = PolicyRetrievalEvaluator()
    # get a fresh loop
    loop = asyncio.new_event_loop()
    # if you want it as the default:
    asyncio.set_event_loop(loop)

    # Stage 1
    loop.run_until_complete(evaluator.preprocess())
    # Stage 2
    loop.run_until_complete(evaluator.run_evaluations())

    # Stage 3
    if asyncio.iscoroutinefunction(evaluator.post_processing):
        result = loop.run_until_complete(evaluator.post_processing())
    else:
        result = evaluator.post_processing()

    summary = json.loads(result) if isinstance(result, str) else result
    yield summary

    loop.close()


@pytest.mark.evaluation
def test_policy_retrieval_hybrid_semantic_ndcg(med_indexer_summary):
    case = next(
        c for c in med_indexer_summary["cases"]
        if "policy-retrieval-hybrid-semantic-001" in c["case"]
    )
    ndcg = case["results"]["metrics"]["NDCGEvaluator.NDCG@10"]
    assert ndcg >= 0.70, f"Expected NDCG ≥ 0.7, got {ndcg}"


@pytest.mark.evaluation
def test_policy_retrieval_hybrid_semantic_recall(med_indexer_summary):
    case = next(
        c for c in med_indexer_summary["cases"]
        if "policy-retrieval-hybrid-semantic-001" in c["case"]
    )
    recall = case["results"]["metrics"]["SearchRecallEvaluator.Recall@10"]
    assert recall >= 0.70, f"Expected Recall ≥ 0.7, got {recall}"

/Users/marcjimz/Documents/Development/aihlsignited-medevals/venv/lib/python3.9/site-packages/pytest_asyncio/plugin.py:217: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
The event loop scope for asynchronous fixtures will default to the fixture caching scope. Future versions of pytest-asyncio will default the loop scope for asynchronous fixtures to function scope. Set the default fixture loop scope explicitly in order to avoid unexpected behavior in the future. Valid fixture loop scopes are: "function", "class", "module", "package", "session"

  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))
2025-04-27 17:31:28,074 - AIFoundryManager - MainProcess - INFO     Configuration validation successful. (aifoundry_helper.py:_validate_configurations:58)
INFO:AIFoundryManager:Configuration validation successful.
2025-04-27 17:31:28,075 - AIFoundryManager - MainProcess - INFO     AIProjectClient initialized successfully

..                                                                                           [100%]
========================================= warnings summary =========================================
../venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1277
  /Users/marcjimz/Documents/Development/aihlsignited-medevals/venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1277: PytestAssertRewriteWarning: Module already imported so cannot be rewritten: anyio
    self._mark_plugins_for_rewrite(hook)

../venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1277
  /Users/marcjimz/Documents/Development/aihlsignited-medevals/venv/lib/python3.9/site-packages/_pytest/config/__init__.py:1277: PytestAssertRewriteWarning: Module already imported so cannot be rewritten: pytest_asyncio
    self._mark_plugins_for_rewrite(hook)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
2 passed, 2 warnings in 15.63s


The above two test cases are examples of passing, here is an example of a failing test case:

In [43]:
%%ipytest

@pytest.mark.evaluation
def test_policy_retrieval_hybrid_semantic_recall(med_indexer_summary):
    case = next(
        c for c in med_indexer_summary["cases"]
        if "policy-retrieval-hybrid-semantic-001" in c["case"]
    )
    recall = case["results"]["metrics"]["SearchRecallEvaluator.Recall@10"]
    assert recall >= 10.00, f"Expected Recall ≥ 10.00, got {recall}"

/Users/marcjimz/Documents/Development/aihlsignited-medevals/venv/lib/python3.9/site-packages/pytest_asyncio/plugin.py:217: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
The event loop scope for asynchronous fixtures will default to the fixture caching scope. Future versions of pytest-asyncio will default the loop scope for asynchronous fixtures to function scope. Set the default fixture loop scope explicitly in order to avoid unexpected behavior in the future. Valid fixture loop scopes are: "function", "class", "module", "package", "session"

  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))
2025-04-27 17:33:13,200 - AIFoundryManager - MainProcess - INFO     Configuration validation successful. (aifoundry_helper.py:_validate_configurations:58)
INFO:AIFoundryManager:Configuration validation successful.
2025-04-27 17:33:13,201 - AIFoundryManager - MainProcess - INFO     AIProjectClient initialized successfully

F                                                                                            [100%]
============================================= FAILURES =============================================
___________________________ test_policy_retrieval_hybrid_semantic_recall ___________________________

med_indexer_summary = {'cases': [{'case': 'policy-retrieval-hybrid-semantic-001', 'results': {'metrics': {'NDCGEvaluator.NDCG@10': 0.7429491...resourceGroups/rg-autoauth-eastus2-autoauth-ui-fix/providers/Microsoft.MachineLearningServices/workspaces/medevals'}}]}

    @pytest.mark.evaluation
    def test_policy_retrieval_hybrid_semantic_recall(med_indexer_summary):
        case = next(
            c for c in med_indexer_summary["cases"]
            if "policy-retrieval-hybrid-semantic-001" in c["case"]
        )
        recall = case["results"]["metrics"]["SearchRecallEvaluator.Recall@10"]
>       assert recall >= 10.00, f"Expected Recall ≥ 10.00, got {recall}"
E       AssertionError: Expe

Now we can define these benchmarks as part of our test suite inclusive of evaluations, unit tests and other integration tests. Feel free to try defining your own benchmarks, or adding your own evaluation and then added your own benchmark. Be sure if you are running it inside the notebook to tag with the `%%ipytest` annotation at the top of your cell.

## Integrating `pytest` into CI/CD with GitHub Actions

To bring this home, you can now tie this into your favorite CI/CD tooling of choice. Ours is Github Actions.

You can add a GitHub Actions workflow file under `.github/workflows/python-tests.yml`. This will:

- Trigger on every push and pull request to `main` (or any branch you choose)
- Set up Python, install dependencies, and run `pytest`
- Report success/failure directly in the PR checks

This is an example of a workflow you can try:

```yaml
# .github/workflows/python-tests.yml
name: Python Test Suite

on:
  push:
    branches:
      - main
      - 'release/*'
  pull_request:
    branches:
      - main

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: [3.9, 3.10]

    steps:
      - name: Checkout code
        uses: actions/checkout@v3

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v4
        with:
          python-version: ${{ matrix.python-version }}

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run pytest
        run: |
          pytest --maxfail=1 --disable-warnings -q

      - name: Archive test reports
        if: always()
        uses: actions/upload-artifact@v3
        with:
          name: pytest-report
          path: ./htmlcov  # an example - feel free to use your own example

And that's it! We've went through a series of labs to show you how you can tie evaluations to multi-discipline roles and ultimately have it part of your automation, with a framework to facilitate easy operations to improving, editing, and iterating on evaluations in support of a **reproducible research** process.

Happy evaluating!